In [1]:
%pip install sentence-transformers

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer #loads BERT-based model
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os

print(" Libraries imported!")

c:\Users\diyaa\anaconda3\envs\bert_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Libraries imported!


In [3]:
df_train = pd.read_csv("../data/resume_jd_train_cleaned.csv")
df_test = pd.read_csv("../data/resume_jd_test_cleaned.csv")

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (6240, 7)
Test shape: (1759, 7)


In [4]:
print("Loading Sentence BERT model...")
print("(First time will download ~90MB — please wait ⏳)")

model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Model loaded!")

Loading Sentence BERT model...
(First time will download ~90MB — please wait ⏳)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2595.17it/s]


✅ Model loaded!


In [5]:
print("Generating resume embeddings... ⏳")
train_resume_embeddings = model.encode(
    df_train['resume_clean'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Generating JD embeddings... ⏳")
train_jd_embeddings = model.encode(
    df_train['jd_clean'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

# Same for test set
print("Generating test embeddings... ⏳")
test_resume_embeddings = model.encode(
    df_test['resume_clean'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

test_jd_embeddings = model.encode(
    df_test['jd_clean'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("✅ All embeddings generated!")
print("Train resume embeddings shape:", train_resume_embeddings.shape)
print("Train JD embeddings shape:    ", train_jd_embeddings.shape)

Generating resume embeddings... ⏳


Batches: 100%|██████████| 195/195 [25:49<00:00,  7.94s/it]   


Generating JD embeddings... ⏳


Batches: 100%|██████████| 195/195 [07:44<00:00,  2.38s/it]


Generating test embeddings... ⏳


Batches: 100%|██████████| 55/55 [01:49<00:00,  1.99s/it]

✅ All embeddings generated!
Train resume embeddings shape: (6240, 384)
Train JD embeddings shape:     (6240, 384)


In [6]:
print("Computing BERT similarity scores...")

# Compute similarity for each resume-JD pair
train_bert_scores = []
for i in range(len(train_resume_embeddings)):
    score = cosine_similarity(
        train_resume_embeddings[i].reshape(1, -1),
        train_jd_embeddings[i].reshape(1, -1)
    )[0][0]
    train_bert_scores.append(score)

test_bert_scores = []
for i in range(len(test_resume_embeddings)):
    score = cosine_similarity(
        test_resume_embeddings[i].reshape(1, -1),
        test_jd_embeddings[i].reshape(1, -1)
    )[0][0]
    test_bert_scores.append(score)

df_train['bert_similarity'] = train_bert_scores
df_test['bert_similarity']  = test_bert_scores

print(" BERT similarity computed!")
print("\nBERT Similarity Stats:")
print(df_train['bert_similarity'].describe())

Computing BERT similarity scores...
 BERT similarity computed!

BERT Similarity Stats:
count    6240.000000
mean        0.613623
std         0.124154
min         0.113965
25%         0.532588
50%         0.625396
75%         0.705670
max         0.912005
Name: bert_similarity, dtype: float64


In [7]:
print("=== TF-IDF vs BERT COMPARISON ===\n")

for label in ['Good Fit', 'Potential Fit', 'No Fit']:
    subset = df_train[df_train['label'] == label]
    tfidf_avg = subset['cosine_similarity'].mean()
    bert_avg  = subset['bert_similarity'].mean()
    print(f"{label}:")
    print(f"  TF-IDF avg: {tfidf_avg:.4f}")
    print(f"  BERT avg:   {bert_avg:.4f}")
    print()

=== TF-IDF vs BERT COMPARISON ===

Good Fit:
  TF-IDF avg: 0.1082
  BERT avg:   0.6551

Potential Fit:
  TF-IDF avg: 0.0994
  BERT avg:   0.6223

No Fit:
  TF-IDF avg: 0.0746
  BERT avg:   0.5890



In [8]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TF-IDF boxplot
import seaborn as sns
sns.boxplot(
    data=df_train, 
    x='label', 
    y='cosine_similarity',
    palette=['green', 'red', 'orange'],
    ax=axes[0]
)
axes[0].set_title('TF-IDF Cosine Similarity')
axes[0].set_xlabel('Match Label')
axes[0].set_ylabel('Similarity Score')

# BERT boxplot
sns.boxplot(
    data=df_train,
    x='label',
    y='bert_similarity',
    palette=['green', 'red', 'orange'],
    ax=axes[1]
)
axes[1].set_title('BERT Semantic Similarity')
axes[1].set_xlabel('Match Label')
axes[1].set_ylabel('Similarity Score')

plt.tight_layout()
plt.savefig('../data/tfidf_vs_bert.png')
plt.show()

print("✅ Comparison plot saved!")

ModuleNotFoundError: No module named 'matplotlib'

BERT semantic similarity scores show 
a larger separation between Good Fit 
(0.6551) and No Fit (0.5890) compared 
to TF-IDF (0.1082 vs 0.0745).

This wider gap indicates that BERT 
embeddings better capture the semantic 
difference between matching and 
non-matching resume-JD pairs, making 
it more effective for classification 
than keyword-based TF-IDF matching.

In [ ]:
os.makedirs("../models", exist_ok=True)

# Save embeddings
np.save("../data/train_resume_embeddings.npy", 
         train_resume_embeddings)
np.save("../data/train_jd_embeddings.npy",     
         train_jd_embeddings)
np.save("../data/test_resume_embeddings.npy",  
         test_resume_embeddings)
np.save("../data/test_jd_embeddings.npy",      
         test_jd_embeddings)

# Save updated CSVs
df_train.to_csv(
    "../data/resume_jd_train_cleaned.csv", index=False)
df_test.to_csv(
    "../data/resume_jd_test_cleaned.csv",  index=False)

# Save BERT model reference
joblib.dump(model, "../models/bert_model.pkl")

print("✅ Everything saved!")
print("\nFiles saved:")
print("  data/train_resume_embeddings.npy")
print("  data/train_jd_embeddings.npy")
print("  data/test_resume_embeddings.npy")
print("  data/test_jd_embeddings.npy")
print("  models/bert_model.pkl")

✅ Everything saved!

Files saved:
  data/train_resume_embeddings.npy
  data/train_jd_embeddings.npy
  data/test_resume_embeddings.npy
  data/test_jd_embeddings.npy
  models/bert_model.pkl


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

model = SentenceTransformer('all-MiniLM-L6-v2')

# =====================
# EXAMPLE 1
# Same meaning, different words
# =====================
print("=" * 50)
print("EXAMPLE 1: Same meaning, different words")
print("=" * 50)

resume1 = "Python developer with machine learning experience"
jd1     = "Looking for ML engineer with Python skills"

r1_emb = model.encode([resume1])
j1_emb = model.encode([jd1])

tfidf_words_resume = set(resume1.lower().split())
tfidf_words_jd     = set(jd1.lower().split())
common_words       = tfidf_words_resume & tfidf_words_jd

bert_score = cosine_similarity(r1_emb, j1_emb)[0][0]

print(f"Resume: {resume1}")
print(f"JD:     {jd1}")
print(f"Common words (TF-IDF sees): {common_words}")
print(f"BERT similarity score:       {bert_score:.4f}")
print("BERT understands: Python developer ≈ ML engineer ✅")

# =====================
# EXAMPLE 2
# Completely different domain
# =====================
print("\n" + "=" * 50)
print("EXAMPLE 2: Completely different domain")
print("=" * 50)

resume2 = "Python developer with machine learning experience"
jd2     = "Looking for experienced nurse with patient care skills"

r2_emb = model.encode([resume2])
j2_emb = model.encode([jd2])

bert_score2 = cosine_similarity(r2_emb, j2_emb)[0][0]

print(f"Resume: {resume2}")
print(f"JD:     {jd2}")
print(f"BERT similarity score: {bert_score2:.4f}")
print("BERT understands: Completely different fields ❌")

# =====================
# EXAMPLE 3
# Partial match
# =====================
print("\n" + "=" * 50)
print("EXAMPLE 3: Partial match")
print("=" * 50)

resume3 = "Junior Python developer 1 year experience"
jd3     = "Senior Python developer 5 years experience required"

r3_emb = model.encode([resume3])
j3_emb = model.encode([jd3])

bert_score3 = cosine_similarity(r3_emb, j3_emb)[0][0]

print(f"Resume: {resume3}")
print(f"JD:     {jd3}")
print(f"BERT similarity score: {bert_score3:.4f}")
print("BERT understands: Same field but different level 🟡")

# =====================
# SUMMARY TABLE
# =====================
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)

print(f"""
Pair                          Score    Label
─────────────────────────────────────────────
Python dev vs ML Engineer     {bert_score:.4f}   Good Fit ✅
Python dev vs Nurse           {bert_score2:.4f}   No Fit ❌
Junior dev vs Senior role     {bert_score3:.4f}   Potential Fit 🟡
""")

Loading weights: 100%|█████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 9514.66it/s]


EXAMPLE 1: Same meaning, different words
Resume: Python developer with machine learning experience
JD:     Looking for ML engineer with Python skills
Common words (TF-IDF sees): {'with', 'python'}
BERT similarity score:       0.7119
BERT understands: Python developer ≈ ML engineer ✅

EXAMPLE 2: Completely different domain
Resume: Python developer with machine learning experience
JD:     Looking for experienced nurse with patient care skills
BERT similarity score: 0.2757
BERT understands: Completely different fields ❌

EXAMPLE 3: Partial match
Resume: Junior Python developer 1 year experience
JD:     Senior Python developer 5 years experience required
BERT similarity score: 0.8879
BERT understands: Same field but different level 🟡

SUMMARY

Pair                          Score    Label
─────────────────────────────────────────────
Python dev vs ML Engineer     0.7119   Good Fit ✅
Python dev vs Nurse           0.2757   No Fit ❌
Junior dev vs Senior role     0.8879   Potential Fit 🟡



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

# =====================
# Define same examples
# =====================
examples = [
    {
        "name": "Same meaning different words",
        "resume": "Python developer with machine learning experience",
        "jd":     "Looking for ML engineer with Python skills",
        "expected": "Good Fit ✅"
    },
    {
        "name": "Completely different domain",
        "resume": "Python developer with machine learning experience",
        "jd":     "Looking for experienced nurse with patient care skills",
        "expected": "No Fit ❌"
    },
    {
        "name": "Partial match",
        "resume": "Junior Python developer 1 year experience",
        "jd":     "Senior Python developer 5 years experience required",
        "expected": "Potential Fit 🟡"
    }
]

# =====================
# TF-IDF Scores
# =====================
# Fit TF-IDF on all example texts
all_texts = [e["resume"] for e in examples] + \
            [e["jd"] for e in examples]

tfidf_temp = TfidfVectorizer()
tfidf_temp.fit(all_texts)

# =====================
# Print Comparison
# =====================
print("=" * 65)
print(f"{'Example':<30} {'TF-IDF':>10} {'BERT':>10} {'Expected'}")
print("=" * 65)

for ex in examples:
    # TF-IDF score
    r_vec = tfidf_temp.transform([ex["resume"]])
    j_vec = tfidf_temp.transform([ex["jd"]])
    tfidf_score = cosine_similarity(r_vec, j_vec)[0][0]
    
    # BERT score
    r_emb = model.encode([ex["resume"]])
    j_emb = model.encode([ex["jd"]])
    bert_score = cosine_similarity(r_emb, j_emb)[0][0]
    
    print(f"{ex['name']:<30} {tfidf_score:>10.4f} "
          f"{bert_score:>10.4f} {ex['expected']}")

print("=" * 65)

# =====================
# Detailed breakdown
# =====================
print("\n=== DETAILED BREAKDOWN ===\n")

for ex in examples:
    print(f" {ex['name'].upper()}")
    print(f"   Resume: {ex['resume']}")
    print(f"   JD:     {ex['jd']}")
    
    # Common words
    resume_words = set(ex['resume'].lower().split())
    jd_words     = set(ex['jd'].lower().split())
    common       = resume_words & jd_words
    print(f"   Common words TF-IDF sees: {common if common else 'NONE'}")
    
    # Scores
    r_vec      = tfidf_temp.transform([ex['resume']])
    j_vec      = tfidf_temp.transform([ex['jd']])
    tfidf_s    = cosine_similarity(r_vec, j_vec)[0][0]
    r_emb      = model.encode([ex['resume']])
    j_emb      = model.encode([ex['jd']])
    bert_s     = cosine_similarity(r_emb, j_emb)[0][0]
    
    print(f"   TF-IDF score: {tfidf_s:.4f}")
    print(f"   BERT score:   {bert_s:.4f}")
    print(f"   Expected:     {ex['expected']}")
    
    # Who got it right?
    if ex['expected'] == "Good Fit ✅":
        tfidf_correct = tfidf_s > 0.3
        bert_correct  = bert_s > 0.5
    elif ex['expected'] == "No Fit ❌":
        tfidf_correct = tfidf_s < 0.1
        bert_correct  = bert_s < 0.3
    else:
        tfidf_correct = 0.1 < tfidf_s < 0.3
        bert_correct  = 0.3 < bert_s < 0.7
    
    print(f"   TF-IDF correct? {'✅' if tfidf_correct else '❌'}")
    print(f"   BERT correct?   {'✅' if bert_correct else '❌'}")
    print()

Example                            TF-IDF       BERT Expected
Same meaning different words       0.1749     0.7119 Good Fit ✅
Completely different domain        0.0855     0.2757 No Fit ❌
Partial match                      0.2817     0.8879 Potential Fit 🟡

=== DETAILED BREAKDOWN ===

 SAME MEANING DIFFERENT WORDS
   Resume: Python developer with machine learning experience
   JD:     Looking for ML engineer with Python skills
   Common words TF-IDF sees: {'with', 'python'}
   TF-IDF score: 0.1749
   BERT score:   0.7119
   Expected:     Good Fit ✅
   TF-IDF correct? ❌
   BERT correct?   ✅

 COMPLETELY DIFFERENT DOMAIN
   Resume: Python developer with machine learning experience
   JD:     Looking for experienced nurse with patient care skills
   Common words TF-IDF sees: {'with'}
   TF-IDF score: 0.0855
   BERT score:   0.2757
   Expected:     No Fit ❌
   TF-IDF correct? ✅
   BERT correct?   ✅

 PARTIAL MATCH
   Resume: Junior Python developer 1 year experience
   JD:     Senior Pytho